# Example 10.6 / 1: Boundary-safe AMPL model

Companion notebook for **10.6 Multi-Period and State-Transition Models** in the AMPL lab Documentation, section "Boundary-safe AMPL model". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Boundary-safe AMPL model

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PERIODS ordered;
param demand {PERIODS} >= 0;
param capacity {PERIODS} >= 0;
param initial_inventory >= 0;
param terminal_inventory >= 0;
param production_cost {PERIODS} >= 0;

var Produce {t in PERIODS} >= 0, <= capacity[t];
var Inventory {PERIODS} >= 0;

minimize TotalCost:
    sum {t in PERIODS} production_cost[t] * Produce[t];

subject to InventoryBalance {t in PERIODS}:
    Inventory[t] =
        (if ord(t) = 1 then initial_inventory else Inventory[prev(t)])
      + Produce[t] - demand[t];

subject to TerminalRequirement:
    Inventory[last(PERIODS)] >= terminal_inventory;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PERIODS"] = ["week1", "week2", "week3"]
ampl.param["demand"] = {"week1": 4, "week2": 6, "week3": 5}
ampl.param["capacity"] = {"week1": 8, "week2": 5, "week3": 8}
ampl.param["initial_inventory"] = 2
ampl.param["terminal_inventory"] = 1
ampl.param["production_cost"] = {"week1": 1, "week2": 3, "week3": 2}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)
print(ampl.var["Inventory"].to_dict())

HiGHS 1.14.0: optimal solution; objective 20
0 simplex iterations
0 barrier iterations
solved
{'week1': 6, 'week2': 0, 'week3': 1}


### Pure AMPL: data.dat and model.run

The same in pure AMPL:

In [6]:
%%writefile data.dat
set PERIODS := week1 week2 week3;
param demand := week1 4 week2 6 week3 5;
param capacity := week1 8 week2 5 week3 8;
param initial_inventory := 2;
param terminal_inventory := 1;
param production_cost := week1 1 week2 3 week3 2;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Produce, Inventory;

HiGHS 1.14.0: optimal solution; objective 20
0 simplex iterations
0 barrier iterations
:     Produce Inventory    :=
week1     8        6
week2     0        0
week3     6        1
;

